In [ ]:
import sys
sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/start_here/')
import pipeline_fxn_lib as lib

import os
import yaml
import shutil
import time

In [ ]:
test_ids = ['110T205L25', '110T218L25', '110T225L25', '130T205L25', '130T218L25', '130T225L25']
num_sims = len(test_ids)
TRANSPORT_TIMESTEP = 5 # Minutes
ICE_GROWTH_TIMESTEP = 1 # Minutes

TEMP_PERTURB = False  # Whether to enable temperature perturbation
TEMP_AMP = 2      # Amplitude of temperature perturbation [K]
TEMP_SCALE = TRANSPORT_TIMESTEP     # Timescale for temperature perturbation in minutes, this is automatically set to transport timestep in APCEMM if they are not the same
SEED_VALUE = 3      # Seed value for random number generation

# Where is the base YAML file located?
source_path = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/B767_LES_CoCiP_APCEMM_input.yaml"  # Source YAML file

# Where do you want to save the modified YAML files and simulation outputs?
test_identifier = f'{TEMP_SCALE}min_no_TP' #f'{TEMP_SCALE}min_TP_{TEMP_AMP}K_s{SEED_VALUE}'
save_directory = f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/testing/{test_identifier}/"

# Where are the base netCDF meteorological files and bypass files located?
base_file_dir = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/"

In [ ]:
# Make YAML files point to the correct .nc file
YAML_paths = []
overlay_paths = []

for i in range(num_sims):
    test_id = test_ids[i]
    print(f"Processing test ID: {test_id}")

    ## For input YAML
    base_destination_dir = f"{save_directory}/{test_id}"  # Target directory
    YAML_destination_path = os.path.join(base_destination_dir, "B767_LES_CoCiP_APCEMM_input.yaml")
    print(f"Destination path: {YAML_destination_path}")

    # Create the destination directory if it doesn't exist
    if not os.path.exists(base_destination_dir):
        print(f"Creating directory: {base_destination_dir}")
        os.makedirs(base_destination_dir, exist_ok=False)

    # Copy the YAML file to the new directory
    shutil.copy(source_path, YAML_destination_path)

    # Read and modify the copied YAML file
    with open(YAML_destination_path, "r") as file:
        data = yaml.safe_load(file)  # Load YAML into a Python dictionary
    
    ## Make an output directory to store results
    output_dir = os.path.join(base_destination_dir, "outputs")
    if not os.path.exists(output_dir):
        print(f"Creating output directory: {output_dir}")
        os.makedirs(output_dir, exist_ok=False)

    # Modify the YAML content
    data["SIMULATION MENU"]["OUTPUT SUBMENU"]["Output folder (string)"] = output_dir
    data["METEOROLOGY MENU"]["METEOROLOGICAL INPUT SUBMENU"]["Met input file path (string)"] = f"{base_file_dir}/{test_id}/{test_id}.nc"
    data["TRANSPORT MENU"]["Transport Timestep [min] (double)"] = TRANSPORT_TIMESTEP
    data["AEROSOL MENU"]["Ice growth timestep [min] (double)"] = ICE_GROWTH_TIMESTEP
    
    if TEMP_PERTURB:
        data["METEOROLOGY MENU"]["TEMPERATURE PERTURBATION SUBMENU"]["Enable Temp. Pert. (T/F)"] = "T"
        data["METEOROLOGY MENU"]["TEMPERATURE PERTURBATION SUBMENU"]["Temp. Perturb. Amplitude (double)"] = TEMP_AMP
        data["METEOROLOGY MENU"]["TEMPERATURE PERTURBATION SUBMENU"]["Temp. Perturb. Timescale (min)"] = TEMP_SCALE
        data["SIMULATION MENU"]["RANDOM NUMBER GENERATION SUBMENU"]["Force seed value (T/F)"] = "T"
        data["SIMULATION MENU"]["RANDOM NUMBER GENERATION SUBMENU"]["Seed value (positive int)"] = SEED_VALUE
    # Write the modified YAML back to the file
    with open(YAML_destination_path, "w") as file:
        yaml.dump(data, file, default_flow_style=False, indent=4)

    ## For overlay input
    overlay_source_path = f"{base_file_dir}/overlay-input.yaml"
    overlay_destination_dir = base_destination_dir #"f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/epm_bypass/{test_id}"
    overlay_destination_path = os.path.join(overlay_destination_dir, "overlay-input.yaml")

    # Create the overlay destination directory if it doesn't exist
    if not os.path.exists(overlay_destination_dir):
        print(f"Creating directory: {overlay_destination_dir}")
        os.makedirs(overlay_destination_dir, exist_ok=False)

    # Copy the YAML file to the new directory
    shutil.copy(overlay_source_path, overlay_destination_path)

    # Read and modify the copied YAML file
    with open(overlay_destination_path, "r") as file:
        data = yaml.safe_load(file)  # Load YAML into a Python dictionary

    # Modify the YAML content
    data["SIMULATION MENU"]["External EPM NetCDF file"] = f"{base_file_dir}/{test_id}/epm-input.nc"

    # Write the modified YAML back to the file
    with open(overlay_destination_path, "w") as file:
        yaml.dump(data, file, default_flow_style=False, indent=4)

    # Save the YAML and Overlay paths for bash inputs
    YAML_paths.append(YAML_destination_path)
    overlay_paths.append(overlay_destination_path)

In [ ]:
# Run batches of APCEMM on slurm
bash_path = "/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/run_apcemm.sh"

for i in range(num_sims):
    arg1 = YAML_paths[i] + " " + overlay_paths[i]
    print(f"Submitting job for test ID: {test_ids[i]}")
    print(f"Input files are: {arg1}")

    export_args = f"ARG1={arg1}"

    # Update where the slurm output file is saved to
    with open(bash_path, "r") as file:
        bash_lines = file.readlines()

    # Modify the output file path in the bash script
    for j, line in enumerate(bash_lines):
        if "#SBATCH -o" in line:
            bash_lines[j] = f"#SBATCH -o /home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/testing/{test_identifier}/{test_ids[i]}/slurm-%j-out\n"
            break

    # Write the modified bash script back to the file
    with open(bash_path, "w") as file:
        file.writelines(bash_lines)

    # Submit the job and get the job ID
    lib.submit_job_and_get_id(bash_path, "has_args", export_args)

    time.sleep(5)  # Optional: wait a bit before submitting the next job